[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO Loss

Implement the **Group Relative Policy Optimization (GRPO)** loss — a group-wise, baseline-subtracted REINFORCE objective commonly used in RLAIF (reinforcement learning from AI feedback).

Given a batch of log-probabilities, scalar rewards, and group ids (one group per prompt), define the within-group normalized advantages:

$$A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}$$

where \(\bar r_{g(i)}\) and \(\text{std}_{g(i)}\) are the mean and standard deviation of rewards in the group of example \(i\).

The GRPO loss is then the negative advantage-weighted log-probability:

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i)\big].$$

### Signature
```python
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """GRPO loss over a batch.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
```

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import torch.nn.functional as F

In [46]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    ids, indices, counts = torch.unique(group_ids, return_inverse=True, return_counts=True)
    output = torch.scatter_reduce(ids.float(), dim=-1, index=indices, src=rewards, reduce="mean", include_self=False)
    r_bar = torch.gather(output, -1, indices)
    zero_mean_reward = rewards - r_bar
    output = torch.scatter_reduce(ids.float(), dim=-1, index=indices, src=zero_mean_reward*zero_mean_reward, reduce="sum", include_self=False) / counts
    r_var = torch.gather(output, -1, indices)
    advantage = zero_mean_reward / (torch.sqrt(r_var) + eps)
    return -(logps * advantage.detach()).mean()
    pass  # compute normalized advantages per group and return -mean(adv.detach() * logps)

In [52]:
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    _, indices, counts = torch.unique(group_ids, return_inverse=True, return_counts=True)
    G = counts.numel()
    buf = lambda: torch.zeros(G, dtype=rewards.dtype, device=rewards.device)
    group_mean = buf().scatter_reduce_(0, index=indices, src=rewards, reduce="mean", include_self=False)
    centered_rewards = rewards - group_mean[indices]
    group_var = buf().scatter_reduce_(0, index=indices, src=centered_rewards*centered_rewards, reduce="mean", include_self=False)
    advantage = centered_rewards / (torch.sqrt(group_var[indices]) + eps)
    return -(logps * advantage.detach()).mean()
    pass  # compute normalized advantages per group and return -mean(adv.detach() * logps)

In [53]:
# 🧪 Debug
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
# group_ids = torch.tensor([1, 0, 3, 0])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

Loss: -0.24997496604919434


In [54]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_loss')


🧪 Testing: GRPO (Group Relative Policy Optimization) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Basic shape & type (2.0ms)
  ✅ [2/4] Numeric check vs reference (4.5ms)
  ✅ [3/4] Gradient flows to logps only (1.7ms)
  ✅ [4/4] Group-wise normalization (0.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (9.0ms total)
  Progress saved. Run status() to see your dashboard.



In [66]:
def grpo_loss(log_probs, log_probs_old, log_probs_ref, advantages, action_mask,
              clip_eps_lo, clip_eps_hi, beta, kl_estimator="kl3"):
    # log_probs        : (G, S) current policy logps (differentiable)
    # log_probs_old    : (G, S) frozen rollout logps (the clip's reference point)
    # log_probs_ref    : (G, S) frozen reference-model logps (the KL's reference point)
    # advantages       : (G, 1) group-standardized, ALREADY normalized -> broadcasts to (G, S)
    # ratio = exp(log_probs - log_probs_old)
    # policy_loss = -min(ratio*A, clamp(ratio, 1-lo, 1+hi)*A)
    # kl   = get_approx_kl(kl_estimator, log_probs, log_probs_ref, mask) if beta else 0   # masked, k3
    # loss = masked_mean(policy_loss + beta*kl, mask, dim=-1).mean(dim=0)
    ...
    rho = torch.exp(log_probs - log_probs_old.detach()) * action_mask
    policy_loss = -torch.minimum(rho * advantages.detach(), rho.clamp(1 - clip_eps_lo, 1 + clip_eps_hi) * advantages.detach())
    log_ratio_ref = (log_probs - log_probs_ref.detach()) * action_mask
    kl = torch.exp(log_ratio_ref) - 1 - log_ratio_ref
    batch_loss = torch.sum((policy_loss + beta * kl ) * action_mask, dim=-1, keepdim=True) / torch.sum(action_mask, dim=-1, keepdim=True)
    loss = batch_loss.mean()
    return loss

In [67]:
import torch, math

def _test_grpo_loss():
    mask = torch.tensor([[1.,1.,1.,0.],[1.,1.,1.,1.]])   # ragged: G=2, lens 3 and 4
    A    = torch.tensor([[0.6],[0.2]])                    # (G,1) per-sequence advantage
    z    = torch.zeros(2,4)

    # ---- Test 1: scalar, finite ----
    L = grpo_loss(z, z, z, A, mask, 0.2, 0.2, 0.0)
    assert L.dim()==0 and L.isfinite(), L
    print("✓ 1: scalar output")

    # ---- Test 2: anchor. ratio=1, beta=0 => A broadcasts to every token, per-seq mean of a
    #      constant is that constant, so loss = -mean_g(A_g) = -(0.6+0.2)/2 = -0.4. ----
    L = grpo_loss(z, z, z, A, mask, 0.2, 0.2, beta=0.0)
    assert torch.allclose(L, torch.tensor(-0.4), atol=1e-6), L
    print("✓ 2: anchor -mean(advantage); (G,1) broadcasts across tokens")

    # ---- Test 3: KL off-switch. policy==old==ref => ratio=1 and kl3(0)=0, so beta must
    #      not change the loss even when > 0. ----
    L0 = grpo_loss(z, z, z, A, mask, 0.2, 0.2, beta=0.0)
    Lk = grpo_loss(z, z, z, A, mask, 0.2, 0.2, beta=0.5)
    assert torch.allclose(L0, Lk, atol=1e-6), (L0, Lk)
    print("✓ 3: KL vanishes when policy==reference, any beta")

    # ---- Test 4: KL isolated & exact. A=0 kills the policy term; set logp-ref=1 so
    #      k3 = e^1-1-1 = e-2, and loss = beta*(e-2). ----
    m1 = torch.tensor([[1.]])
    L = grpo_loss(torch.tensor([[0.]]), torch.tensor([[0.]]), torch.tensor([[-1.]]),
                  torch.tensor([[0.]]), m1, 0.2, 0.2, beta=0.5, kl_estimator="kl3")
    assert torch.allclose(L, torch.tensor(0.5*(math.e - 2)), atol=1e-6), L
    print("✓ 4: KL term is exact and lives in the loss (k3)")

    # ---- Test 5: masking. Perturb log_probs & ref ONLY at padded positions; loss unchanged. ----
    lp = torch.randn(2,4); lpo = lp.clone(); ref = torch.randn(2,4)
    base = grpo_loss(lp, lpo, ref, A, mask, 0.2, 0.2, beta=0.5)
    lp2, ref2 = lp.clone(), ref.clone()
    lp2[mask==0] += 2.0; ref2[mask==0] += 2.0
    pert = grpo_loss(lp2, lpo, ref2, A, mask, 0.2, 0.2, beta=0.5)
    assert torch.allclose(base, pert, atol=1e-6) and pert.isfinite(), (base, pert)
    print("✓ 5: padding excluded from the loss")

    # ---- Test 6: policy clip caps upside. A>0, ratio=1.5 -> clipped to 1.2. ----
    L = grpo_loss(torch.tensor([[math.log(1.5)]]), torch.zeros(1,1), torch.zeros(1,1),
                  torch.tensor([[1.]]), m1, 0.2, 0.2, beta=0.0)
    assert torch.allclose(L, torch.tensor(-1.2), atol=1e-5), L
    print("✓ 6: one-directional clip caps the surrogate at the band edge")

    # ---- Test 7: reduction is per-sequence-then-batch, not a global token mean. ----
    mR = torch.tensor([[1.,0.,0.,0.],[1.,1.,1.,1.]])       # lens 1 and 4
    AR = torch.tensor([[4.],[1.]])
    L = grpo_loss(torch.zeros(2,4), torch.zeros(2,4), torch.zeros(2,4), AR, mR, 0.2, 0.2, beta=0.0)
    # per-seq: -4 and -1 -> batch mean -2.5 ; global token mean would be (-4-1-1-1-1)/5 = -1.6
    assert torch.allclose(L, torch.tensor(-2.5), atol=1e-6), L
    assert abs(L.item() - (-1.6)) > 1e-3
    print("✓ 7: per-sequence-then-batch reduction")

    print("\nAll grpo_loss tests passed.")

_test_grpo_loss()

✓ 1: scalar output
✓ 2: anchor -mean(advantage); (G,1) broadcasts across tokens
✓ 3: KL vanishes when policy==reference, any beta
✓ 4: KL term is exact and lives in the loss (k3)
✓ 5: padding excluded from the loss
✓ 6: one-directional clip caps the surrogate at the band edge
✓ 7: per-sequence-then-batch reduction

All grpo_loss tests passed.
